# PL ?????????v3?

? notebook ??? PYNQ ????? bitstream ?????
- AXI-Lite ????
- DMA ?? + TLAST ??
- ? tile / ? tile / ? N tile ????
- ?????????

????????????? **????** ??????/???

In [1]:
# v3
print('PL Validate Notebook v3')

import time
import numpy as np
import pynq


PL Validate Notebook v3


In [2]:
# -----------------------------
# ?????????
# -----------------------------
ARRAY_ROW = 12
ARRAY_COL = 16

def ceil_to(x, base):
    return ((x + base - 1) // base) * base

def calc_pad(m, k, n):
    m_pad = m if (m % 2 == 0) else (m + 1)
    k_pad = ceil_to(k, ARRAY_ROW)
    n_pad = ceil_to(n, ARRAY_COL)
    return m_pad, k_pad, n_pad

def to_u8(x):
    return int(x) & 0xFF

def u8_to_i8(arr_u8):
    arr_u8 = np.array(arr_u8, dtype=np.uint8)
    return np.where(arr_u8 < 128, arr_u8, arr_u8 - 256).astype(np.int8)

def ppu_model(x, mult, shift, zp, bias):
    # x int32
    val = (x + bias) * mult
    val = val >> shift
    val = val + zp
    val = np.clip(val, -128, 127)
    return val.astype(np.int8)

def pack_input_tile(a_tile):
    # a_tile: [M_PAD, 12] int8
    bytes_list = []
    for r in range(a_tile.shape[0]):
        for c in range(ARRAY_ROW):
            bytes_list.append(to_u8(a_tile[r, c]))
    words = []
    for i in range(0, len(bytes_list), 8):
        w = 0
        for b_i in range(8):
            w |= (bytes_list[i + b_i] << (8 * b_i))
        words.append(np.uint64(w))
    return words

def pack_weight_tile(b_tile):
    # b_tile: [12, 16] int8
    words = []
    for r in range(ARRAY_ROW):
        val128 = 0
        for c in range(ARRAY_COL):
            val = to_u8(b_tile[r, c])
            val128 |= (val << (8 * c))
        low = val128 & 0xFFFFFFFFFFFFFFFF
        high = (val128 >> 64) & 0xFFFFFFFFFFFFFFFF
        words.append(np.uint64(low))
        words.append(np.uint64(high))
    return words

def unpack_output_tile(words, m_pad):
    # words length = m_pad * 2
    out = np.zeros((m_pad, ARRAY_COL), dtype=np.int8)
    for r in range(m_pad):
        low = int(words[2*r])
        high = int(words[2*r + 1])
        val128 = low | (high << 64)
        bytes_row = [(val128 >> (8*c)) & 0xFF for c in range(ARRAY_COL)]
        out[r, :] = u8_to_i8(bytes_row)
    return out

def make_case(m, k, n, seed=1, pattern='random'):
    m_pad, k_pad, n_pad = calc_pad(m, k, n)
    if pattern == 'ones':
        a = np.ones((m, k), dtype=np.int8)
        b = (np.ones((k, n), dtype=np.int8) * 2)
    else:
        np.random.seed(seed)
        a = np.random.randint(-10, 10, size=(m, k), dtype=np.int8)
        b = np.random.randint(-10, 10, size=(k, n), dtype=np.int8)
    a_pad = np.zeros((m_pad, k_pad), dtype=np.int8)
    b_pad = np.zeros((k_pad, n_pad), dtype=np.int8)
    a_pad[:m, :k] = a
    b_pad[:k, :n] = b
    return a_pad, b_pad, m_pad, k_pad, n_pad


In [3]:
# -----------------------------
# ????????
# -----------------------------
BITSTREAM = './deit/deit_accel.bit'
IP_NAME = 'deit_accelerator_top_0'
DMA_NAME = 'axi_dma_0'

PPU_CFG = {
    'mult': 180,
    'shift': 8,
    'zp': 10,
    'bias': 100,
}

REG_CTRL      = 0x00
REG_STATUS    = 0x04
REG_CFG_SEQ   = 0x08
REG_CFG_ACC   = 0x0C
REG_VERSION   = 0x10
REG_PPU_MULT  = 0x14
REG_PPU_SHIFT = 0x18
REG_PPU_ZP    = 0x1C
REG_PPU_BIAS  = 0x20
REG_OUT_EN    = 0x24

REG_DBG_SNAP  = 0x28
REG_DBG_CLR   = 0x2C
REG_DBG0      = 0x30
REG_DBG1      = 0x34
REG_DBG2      = 0x38
REG_DBG3      = 0x3C

overlay = pynq.Overlay(BITSTREAM)
axi_ctrl = getattr(overlay, IP_NAME)
dma = getattr(overlay, DMA_NAME)

def axi_write(addr, val):
    axi_ctrl.write(addr, int(val))

def axi_read(addr):
    return axi_ctrl.read(addr)

def soft_reset():
    axi_write(REG_CTRL, 0x00)
    time.sleep(0.01)
    axi_write(REG_CTRL, 0x02)
    time.sleep(0.01)

def start_pulse():
    axi_write(REG_CTRL, 0x03)
    axi_write(REG_CTRL, 0x02)

def dump_regs():
    regs = {
        'CTRL': axi_read(REG_CTRL),
        'STATUS': axi_read(REG_STATUS),
        'CFG_SEQ': axi_read(REG_CFG_SEQ),
        'CFG_ACC': axi_read(REG_CFG_ACC),
        'VERSION': axi_read(REG_VERSION),
        'PPU_MULT': axi_read(REG_PPU_MULT),
        'PPU_SHIFT': axi_read(REG_PPU_SHIFT),
        'PPU_ZP': axi_read(REG_PPU_ZP),
        'PPU_BIAS': axi_read(REG_PPU_BIAS),
        'OUT_EN': axi_read(REG_OUT_EN),
    }
    print('=== AXI-Lite Register Dump ===')
    for k, v in regs.items():
        print(f'{k:>8} = 0x{v:08x}')
    return regs

def dbg_snap():
    axi_write(REG_DBG_SNAP, 1)

def dbg_clr():
    axi_write(REG_DBG_CLR, 1)

def dbg_read():
    dbg_snap()
    d0 = axi_read(REG_DBG0)
    d1 = axi_read(REG_DBG1)
    d2 = axi_read(REG_DBG2)
    d3 = axi_read(REG_DBG3)
    info = {
        'DBG0': d0,
        'DBG1': d1,
        'DBG2': d2,
        'DBG3': d3,
        'state': d0 & 0x7,
        'dma_req': (d0 >> 3) & 0x1,
        'w_load_en': (d0 >> 4) & 0x1,
        'in_stream_en': (d0 >> 5) & 0x1,
        'wbuf_valid': (d0 >> 6) & 0x1,
        'ibuf_valid': (d0 >> 7) & 0x1,
        'axis_in_v': (d0 >> 8) & 0x1,
        'axis_out_v': (d0 >> 9) & 0x1,
        'axis_out_last': (d0 >> 10) & 0x1,
        'obuf_full': (d0 >> 11) & 0x1,
    }
    return info

def dma_status(ch):
    return ch._mmio.read(0x04)

def wait_dma_idle(ch, timeout=2.0):
    t0 = time.time()
    while time.time() - t0 < timeout:
        sr = dma_status(ch)
        if sr & 0x2:
            return True, sr
        time.sleep(0.001)
    return False, dma_status(ch)

print('[INFO] Overlay loaded')
dump_regs()


[INFO] Overlay loaded
=== AXI-Lite Register Dump ===
    CTRL = 0x00000000
  STATUS = 0x00000002
 CFG_SEQ = 0x00000000
 CFG_ACC = 0x00000000
 VERSION = 0x20260117
PPU_MULT = 0x00000000
PPU_SHIFT = 0x00000000
  PPU_ZP = 0x00000000
PPU_BIAS = 0x00000000
  OUT_EN = 0x00000000


{'CTRL': 0,
 'STATUS': 2,
 'CFG_SEQ': 0,
 'CFG_ACC': 0,
 'VERSION': 539361559,
 'PPU_MULT': 0,
 'PPU_SHIFT': 0,
 'PPU_ZP': 0,
 'PPU_BIAS': 0,
 'OUT_EN': 0}

In [4]:
# -----------------------------
# ????? tile ?? A / B???? C
# -----------------------------
def run_case(m, k, n, seed=1, pattern='random', timeout=2.0, verbose=True):
    a_pad, b_pad, m_pad, k_pad, n_pad = make_case(m, k, n, seed=seed, pattern=pattern)
    k_tiles = k_pad // ARRAY_ROW
    n_tiles = n_pad // ARRAY_COL

    if verbose:
        print(f'>>> CASE: M={m} K={k} N={n} | M_PAD={m_pad} K_PAD={k_pad} N_PAD={n_pad} | K_TILES={k_tiles} N_TILES={n_tiles}')

    # Golden
    c_int32 = a_pad.astype(np.int32) @ b_pad.astype(np.int32)
    c_gold = ppu_model(c_int32, PPU_CFG['mult'], PPU_CFG['shift'], PPU_CFG['zp'], PPU_CFG['bias'])

    # Config
    soft_reset()
    dbg_clr()
    axi_write(REG_CFG_SEQ, m_pad)
    axi_write(REG_CFG_ACC, 0)
    axi_write(REG_PPU_MULT, PPU_CFG['mult'])
    axi_write(REG_PPU_SHIFT, PPU_CFG['shift'])
    axi_write(REG_PPU_ZP, PPU_CFG['zp'])
    axi_write(REG_PPU_BIAS, PPU_CFG['bias'])
    axi_write(REG_OUT_EN, 0)

    # HW output buffer
    c_hw = np.zeros((m_pad, n_pad), dtype=np.int8)

    for n_idx in range(n_tiles):
        for k_idx in range(k_tiles):
            acc_mode = 0 if (k_idx == 0) else 1
            out_en = 1 if (k_idx == k_tiles - 1) else 0
            axi_write(REG_CFG_ACC, acc_mode)
            axi_write(REG_OUT_EN, out_en)

            # Preload input tile in IDLE
            a_tile = a_pad[:, k_idx*ARRAY_ROW:(k_idx+1)*ARRAY_ROW]
            in_words = pack_input_tile(a_tile)
            buf_A = pynq.allocate(shape=(len(in_words),), dtype=np.uint64)
            buf_A[:] = in_words
            buf_A.flush()
            dma.sendchannel.transfer(buf_A)
            ok_mm2s, _ = wait_dma_idle(dma.sendchannel, timeout=timeout)
            if not ok_mm2s:
                print('[FAIL] MM2S not idle after A preload')
                return False

            # If output for this N tile, arm RX before start
            if out_en:
                out_words = m_pad * 2
                buf_C = pynq.allocate(shape=(out_words,), dtype=np.uint64)
                dma.recvchannel.transfer(buf_C)
            else:
                buf_C = None

            # Start
            start_pulse()

            # Wait DMA request high via DBG0[3]
            t0 = time.time()
            while time.time() - t0 < timeout:
                if (dbg_read()['dma_req'] == 1):
                    break
                time.sleep(0.0005)
            else:
                print('[FAIL] DMA_REQ timeout')
                return False

            # Send weight tile
            b_tile = b_pad[k_idx*ARRAY_ROW:(k_idx+1)*ARRAY_ROW, n_idx*ARRAY_COL:(n_idx+1)*ARRAY_COL]
            w_words = pack_weight_tile(b_tile)
            buf_B = pynq.allocate(shape=(len(w_words),), dtype=np.uint64)
            buf_B[:] = w_words
            buf_B.flush()
            dma.sendchannel.transfer(buf_B)
            ok_mm2s2, _ = wait_dma_idle(dma.sendchannel, timeout=timeout)
            if not ok_mm2s2:
                print('[FAIL] MM2S not idle after weight')
                return False

            # Wait AP_DONE
            t0 = time.time()
            done = False
            while time.time() - t0 < timeout:
                if (axi_read(REG_STATUS) & 0x1) != 0:
                    done = True
                    axi_write(REG_STATUS, 0x1)
                    break
                time.sleep(0.0005)
            if not done:
                print('[FAIL] AP_DONE timeout')
                return False

            # Receive output tile
            if out_en:
                ok_s2mm, _ = wait_dma_idle(dma.recvchannel, timeout=timeout)
                if not ok_s2mm:
                    print('[FAIL] S2MM timeout')
                    return False
                buf_C.invalidate()
                out_tile = unpack_output_tile(np.array(buf_C), m_pad)
                c_hw[:, n_idx*ARRAY_COL:(n_idx+1)*ARRAY_COL] = out_tile
                buf_C.close()

            # cleanup
            buf_A.close()
            buf_B.close()

    # Compare
    if np.array_equal(c_hw, c_gold):
        if verbose:
            print('[PASS] ????')
        return True
    else:
        # find first mismatch
        idx = np.argwhere(c_hw != c_gold)[0]
        r, c = int(idx[0]), int(idx[1])
        print(f'[FAIL] mismatch at ({r},{c}) hw={c_hw[r,c]} gold={c_gold[r,c]}')
        return False


In [11]:
# -----------------------------
# ????
# -----------------------------
tests = [
    {'name': 'T1-single-tile-ones', 'm':2, 'k':12, 'n':16, 'seed':1, 'pattern':'ones'},
    {'name': 'T2-single-tile-rand', 'm':3, 'k':12, 'n':16, 'seed':2, 'pattern':'random'},
    {'name': 'T3-multi-K', 'm':4, 'k':24, 'n':16, 'seed':3, 'pattern':'random'},
    {'name': 'T4-multi-N', 'm':4, 'k':12, 'n':32, 'seed':4, 'pattern':'random'},
    # ??????????????????
    {'name': 'T5-case1', 'm':48, 'k':36, 'n':48, 'seed':1, 'pattern':'random'},
    {'name': 'T6-case2', 'm':67, 'k':93, 'n':67, 'seed':2, 'pattern':'random'},
]

results = []
for t in tests:
    print('=== RUN', t['name'], '===')
    ok = run_case(t['m'], t['k'], t['n'], seed=t['seed'], pattern=t['pattern'])
    results.append((t['name'], ok))

print('=== SUMMARY ===')
for name, ok in results:
    print(f'{name}:', 'PASS' if ok else 'FAIL')


=== RUN T1-single-tile-ones ===
>>> CASE: M=2 K=12 N=16 | M_PAD=2 K_PAD=12 N_PAD=16 | K_TILES=1 N_TILES=1
[PASS] ????
=== RUN T2-single-tile-rand ===
>>> CASE: M=3 K=12 N=16 | M_PAD=4 K_PAD=12 N_PAD=16 | K_TILES=1 N_TILES=1
[PASS] ????
=== RUN T3-multi-K ===
>>> CASE: M=4 K=24 N=16 | M_PAD=4 K_PAD=24 N_PAD=16 | K_TILES=2 N_TILES=1
[PASS] ????
=== RUN T4-multi-N ===
>>> CASE: M=4 K=12 N=32 | M_PAD=4 K_PAD=12 N_PAD=32 | K_TILES=1 N_TILES=2
[PASS] ????
=== RUN T5-case1 ===
>>> CASE: M=48 K=36 N=48 | M_PAD=48 K_PAD=36 N_PAD=48 | K_TILES=3 N_TILES=3
[PASS] ????
=== RUN T6-case2 ===
>>> CASE: M=67 K=93 N=67 | M_PAD=68 K_PAD=96 N_PAD=80 | K_TILES=8 N_TILES=5
[PASS] ????
=== SUMMARY ===
T1-single-tile-ones: PASS
T2-single-tile-rand: PASS
T3-multi-K: PASS
T4-multi-N: PASS
T5-case1: PASS
T6-case2: PASS


In [9]:
# -----------------------------
# ???????????
# -----------------------------
def run_two_tasks():
    print('=== RUN TWO TASKS ===')
    ok1 = run_case(48, 36, 48, seed=1, pattern='random', timeout=3.0)
    ok2 = run_case(67, 93, 67, seed=2, pattern='random', timeout=3.0)
    print('[RESULT] Task1:', 'PASS' if ok1 else 'FAIL')
    print('[RESULT] Task2:', 'PASS' if ok2 else 'FAIL')
    return ok1 and ok2

# ????????????
run_two_tasks()


=== RUN TWO TASKS ===
>>> CASE: M=48 K=36 N=48 | M_PAD=48 K_PAD=36 N_PAD=48 | K_TILES=3 N_TILES=3
[PASS] ????
>>> CASE: M=67 K=93 N=67 | M_PAD=68 K_PAD=96 N_PAD=80 | K_TILES=8 N_TILES=5
[PASS] ????
[RESULT] Task1: PASS
[RESULT] Task2: PASS


True